In [1]:
import cartopy.crs as ccrs
import xarray as xr
import pandas as pd
import numpy as np
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from matplotlib import pyplot as plt
import glob
import matplotlib
%matplotlib inline

In [2]:
#define focusing area
[min_lon, max_lon, min_lat, max_lat]=[100.,150.,10.,50.]
[min_lon_tw, max_lon_tw, min_lat_tw, max_lat_tw]=[118,126,22,26]
[min_lon_ish, max_lon_ish, min_lat_ish, max_lat_ish]=[122,126,22,26]
slp_levels=np.linspace(980,1040,16)
slp_lws=[2 if (pp-980)%20==0 else 1 for pp in slp_levels]
prec_threshold=0.1
ishigaki={'lat':24.33,'lon':124.16}

In [3]:
dataDir='/data/mileshsieh/CMIP6'
m='TaiESM1'
sceList=['ssp126','ssp245','ssp370','ssp585']


template='%s/%s/%s/atmos/day/r1i1p1f1/%s_day_%s_%s_r1i1p1f1_gn_%04d0101-*.nc'
varList=['ua','va','pr']
p_selected=1000
yrList=np.arange(2015,2095,10)

In [4]:
def uv2ws(u, v):
    func = lambda x, y: np.sqrt(x**2 + y**2)
    return xr.apply_ufunc(func, u, v, dask="allowed")

def uv2wd(u, v):
    func = lambda x,y: np.where(u==0,np.where(v>0,180.0,0.0),180.0 + np.arctan2(u, v) * 180.0 / np.pi)
    return xr.apply_ufunc(func, u, v, dask="allowed")

def calc_wd_cor(wd_array):
    wd_min=wd_array.min()
    #print(wd_min)
    wdList=np.array([dd-wd_min if abs(dd-wd_min)<180.0 else wd_min-(dd-360.0) for dd in wd_array.ravel()])
    result=np.percentile(wdList,75)
    return result
def uv2deg_ish(ser):
    if ser.ish_u==0:
        return 180.0 if ser.ish_v>0 else 0.0
    else:
        tmpdir=270-np.arctan(ser.ish_v/ser.ish_u)/np.pi*180
        return tmpdir if ser.ish_u>0 else tmpdir-180
    
def uv2deg_mean(ser):
    if ser.meanU==0:
        return 180.0 if ser.meanV>0 else 0.0
    else:
        tmpdir=270-np.arctan(ser.meanV/ser.meanU)/np.pi*180
        return tmpdir if ser.meanU>0 else tmpdir-180

In [6]:
#for sce in sceList:
for sce in ['ssp585',]:
    for yr in [2015,]:
    #for yr in yrList:
        yr_start=yr
        if yr_start==2095:
            yr_end=2100
        else:
            yr_end=yr+9
        #LTS
        ds_model_ta=xr.open_dataset(glob.glob(template%(dataDir,m,sce,'ta',m,sce,yr_start))[0])\
                              .sel(lat=slice(min_lat,max_lat),lon=slice(min_lon,max_lon),plev=[70000,100000])
        ds_model_ta['LTS']=ds_model_ta.ta.sel(plev=70000)*np.power(1000/700,287/1004)-ds_model_ta.ta.sel(plev=100000).compute()
        fList=[glob.glob(template%(dataDir,m,sce,var,m,sce,yr_start))[0] for var in varList]
        print(sce,yr)
        #open multiple files
        ds_model_uv_mslp=xr.open_mfdataset(fList).sel(lat=slice(min_lat,max_lat),lon=slice(min_lon,max_lon),plev=p_selected*100)

        ds_model=xr.merge([ds_model_uv_mslp,ds_model_ta.LTS])
        #convert time coordinate to date
        ds_model.coords['time'] = ds_model.time.dt.floor('1D')

        tw_ds=ds_model.sel(lat=slice(min_lat_tw,max_lat_tw),lon=slice(min_lon_tw,max_lon_tw))
        tw_ds['precipitation']=tw_ds.pr*3600
        tw_ds['prRatio']=tw_ds.precipitation.where(tw_ds.precipitation>=prec_threshold).count(dim=['lon','lat'])/30.0
        df_tw=pd.DataFrame({'date':tw_ds.time.values,'prRatio':tw_ds.prRatio.values})
    
        ish_ds=ds_model.sel(lat=slice(min_lat_ish,max_lat_ish),lon=slice(min_lon_ish,max_lon_ish))
        ish_ds['ws']=uv2ws(ish_ds.ua, ish_ds.va)
        ish_ds['wd']=uv2wd(ish_ds.ua, ish_ds.va)
        #ish_ds['wd_p90']=ish_ds.wd.reduce(np.percentile, q=90,dim=['lat','lon'])
        #ish_ds['wd_p10']=ish_ds.wd.reduce(np.percentile, q=10,dim=['lat','lon'])
        #ish_ds['wdCor_raw']=ish_ds.wd_p90-ish_ds.wd_p10
        #ish_ds['wdCor']=ish_ds.wdCor_raw.where(ish_ds.wdCor_raw<180,360-ish_ds.wdCor_raw)
        ish_ds['wdCor']=xr.apply_ufunc(calc_wd_cor,ish_ds.wd,input_core_dims=[['lat','lon']],
                                   output_core_dims=[[]],vectorize=True)
        ish_ds['ws_mean']=ish_ds.ws.reduce(np.mean,dim=['lat','lon'])
        ish_ds['maxWS']=ish_ds.ws.reduce(np.max,dim=['lat','lon'])
        ish_ds['u_mean']=ish_ds.ua.reduce(np.mean,dim=['lat','lon'])
        ish_ds['v_mean']=ish_ds.va.reduce(np.mean,dim=['lat','lon'])  
        ish_ds['LTS_mean']=ish_ds.LTS.reduce(np.mean,dim=['lat','lon'])  
        idx=ish_ds.wdCor.idxmax(dim='time')
        dateStr=ish_ds.wdCor.idxmax(dim='time').dt.strftime('%Y-%m-%d').item()
        print(dateStr,pd.to_datetime(dateStr).dayofyear,ish_ds.wdCor.max().item())
        #save to dataframe
        yyyymmdd=ish_ds.time.dt.strftime('%Y%m%d')
        u_ish=ish_ds.ua.interp(lat=ishigaki['lat'],lon=ishigaki['lon']).values
        v_ish=ish_ds.va.interp(lat=ishigaki['lat'],lon=ishigaki['lon']).values
        LTS_ish=ish_ds.LTS.interp(lat=ishigaki['lat'],lon=ishigaki['lon']).values
        wdCor=ish_ds.wdCor.values
        df_ish=pd.DataFrame({'date':ish_ds.time.values,'yyyymmdd':yyyymmdd, 
                     'meanU':ish_ds.u_mean.values,'meanV':ish_ds.v_mean.values,'meanLTS':ish_ds.LTS_mean.values,
                     'wdCor':wdCor,'meanWS':ish_ds.ws_mean.values,'maxWS':ish_ds.maxWS.values,
                     'ish_u':u_ish,'ish_v':v_ish,'ish_LTS':LTS_ish,})
        df_ish['meanWD']=df_ish.apply(uv2deg_mean,axis=1)
        df_ish['ish_wd']=df_ish.apply(uv2deg_ish,axis=1)
        df_ish['ish_ws']=df_ish.apply(lambda ser: np.sqrt(ser.ish_u*ser.ish_u+ser.ish_v*ser.ish_v),axis=1)
        df_ish['year']=df_ish.apply(lambda ser: ser.date.year,axis=1)
        df_ish['month']=df_ish.apply(lambda ser: ser.date.month,axis=1)
    
        final=pd.merge(df_ish,df_tw,on='date')
        final[final.month.isin([1,2,3,4,10,11,12])].to_csv('../data/cold_season_%s_%s_%04d0101-%04d1231.indices.csv'%(m,sce,yr_start,yr_end), index=False)


ssp585 2015
2018-09-01 244 174.93603515625
